In [29]:
import torch
from torch import nn

MAX_OTHERS = 128
D_MODEL = 64

NUM_ITEMS = 2 #needs to sync with Rust

In [25]:
from torch._higher_order_ops import scan


def sample_stochastic_gate(logits):
    """
    Continuous stochastic gate in (0, 1).
    """
    u = torch.rand_like(logits).clamp_(1e-6, 1.0 - 1e-6)
    noise = torch.log(u) - torch.log1p(-u)
    return torch.sigmoid(logits + noise)


class AproxMemory(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        output_size,
        K,
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.output_size = output_size
        self.K = K

        # i, f, candidate
        self.x_proj = nn.Linear(input_size, 3 * hidden_size)
        self.h_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)

        self.out_proj = nn.Linear(hidden_size, output_size)

    def forward(self, x, state=None):
        """
        x:
            (T, B, K, D)

        state:
            optional recurrent state:
            (B, K, H)

        returns:
            output:
                (T, B, K, O)

            state:
                final recurrent state
                (B, K, H)

            hidden:
                recurrent state at every timestep
                (T, B, K, H)

            activity:
                mean(i + f) per merchant
                (T, B, K)
        """
        T, B, K, _ = x.shape

        if K != self.K:
            raise ValueError(f"Expected K={self.K}, got K={K}")

        if state is None:
            state = x.new_zeros(B, K, self.hidden_size)

        def step(c, x_t):
            """
            c:
                (B, K, H)

            x_t:
                (B, K, D)
            """

            h = torch.tanh(c)

            # Linear naturally treats B and K as batch dimensions.
            gates = self.x_proj(x_t) + self.h_proj(h)

            i_logits, f_logits, g_logits = gates.chunk(3, dim=-1)

            i = sample_stochastic_gate(i_logits)
            f = sample_stochastic_gate(f_logits)
            g = torch.tanh(g_logits)

            c_new = f * c + i * g
            h_new = torch.tanh(c_new)

            y = self.out_proj(h_new)

            # (B, K)
            activity = (i + f).mean(dim=-1)

            return c_new, (y, h_new, activity)

        state, (output, hidden, activity) = scan(
            step,
            state,
            x,
        )

        return output, state, hidden, activity

In [28]:
AproxMemory(10,10,10,MAX_OTHERS)

AproxMemory(
  (x_proj): Linear(in_features=10, out_features=30, bias=True)
  (h_proj): Linear(in_features=10, out_features=30, bias=False)
  (out_proj): Linear(in_features=10, out_features=10, bias=True)
)

In [ ]:
@dataclass
class ActionParams:
    mode_logits: Tensor

    trade_count_mu: Tensor
    trade_count_log_sigma: Tensor

    barter_k_logits: Tensor
    barter_item_logits: Tensor
    barter_amount_mu: Tensor
    barter_amount_log_sigma: Tensor

    merchant_query: Tensor
    city_query: Tensor

In [35]:
class ActionLayer(nn.Module):
    def __init__(self,in_features):
        super().__init__()
        self.f = nn.Linear(in_features,
                           3+ #which of the 3 game modes to go
                           1+ #trade accept/reject
                           (2+NUM_ITEMS)#sample trade offer
                           +D_MODEL#embedded dim for selecting best fiting merchat
                           +D_MODEL#for city select
                          )
    def forward(self,x,neighbor_cities,city_merchants,barter_offer=None):
        x = self.f(x)

        act = torch.softmax(x[:,:,:3],dim=-1)
        
        city_k = x[:,:,-D_MODEL].unsqeeze(-1)
        merchant_k = x[:,:,-2*D_MODEL::-D_MODEL].unsqeeze(-1)

        city_probs = act[2]*torch.softmax(neighbor_cities@cit_k,dim=-1).squeeze(-1)
        merchant_probs = act[1]*torch.softmax(city_merchants@merchant_k,dim=-1).squeeze(-1)

        accept_prob = act[0]*torch.sigmoid(x[3])
        if barter_offer==None:
            barter_offer = sample_barter(x[:,:,4:2+NUM_ITEMS])
                                         
        barter_probs = act[0]*get_barter_probs(barter_offer,x[:,:,4:2+NUM_ITEMS])

        return barter_probs,barter_offer,merchant_probs,city_probs

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.merchants = AproxMemory(D_MODEL,D_MODEL,D_MODEL,MAX_OTHERS)
    
    def forward(self,x):
        pass

Model()